In [ ]:
import os

import django

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "api.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

os.environ["DATABASE_HOST"] = ""
os.environ["DATABASE_PORT"] = ""
os.environ["DATABASE_PASSWORD"] = ""

os.environ["GOOGLE_API_KEY"] = ""
os.environ["GOOGLE_CUSTOM_SEACH_ID"] = ""

django.setup()

## Release Info


In [ ]:
import datetime

release_beers = []
release_name = "Mai 2025"
release_date = datetime.datetime(year=2025, month=5, day=7, hour=6, minute=0)
badge_text = ""
badge_type = ""
badge_days = 35

## Add new products


In [ ]:
from django.utils import timezone

from beers.models import Beer
from beers.vmp_commands import apply_product_fields
from clients.vmp import VmpApiError, VmpClient

client = VmpClient.from_external_api()

for vmp_id in release_beers:
    try:
        product = client.get_product(vmp_id)
    except VmpApiError as e:
        print(f"Failed to fetch {vmp_id}: {e}")
        continue

    beer, _ = Beer.objects.get_or_create(vmp_id=int(product.code))
    apply_product_fields(beer, product)
    if product.producer is not None:
        beer.vmp_brewery = product.producer.name
    beer.vmp_updated = timezone.now()
    beer.active = True
    beer.save()
    print(f"Added: {beer.vmp_name}")


## Match Beers


In [ ]:
from django.core.management import call_command

from beers.models import Beer

unmatched = Beer.objects.filter(
    untpd_id__isnull=True, match_manually=False, active=True
).count()

call_command("match_untappd", unmatched)


In [ ]:
from datetime import timedelta

from django.utils import timezone

thirty_days_ago = timezone.now() - timedelta(days=30)

recent_beers = Beer.objects.filter(created_at__gte=thirty_days_ago)

for beer in recent_beers:
    beer.active = True
    beer.match_manually = False
    beer.save()

## Update Beers


In [ ]:
from django.core.management import call_command

from beers.models import Beer

count = Beer.objects.filter(untpd_id__isnull=False, active=True).count()

call_command("update_beers_from_untappd", count)


## Add release model


In [ ]:
from beers.models import Release

try:
    release = Release.objects.get(name=release_name)
except Release.DoesNotExist:
    release = Release.objects.create(name=release_name, release_date=release_date)

for b in release_beers:
    try:
        beer = Beer.objects.get(vmp_id=b)
    except Beer.DoesNotExist:
        continue

    release.beer.add(beer.vmp_id)
    release.save()

print("Release added")

## Check Beers


In [ ]:
from datetime import timedelta

from django.utils import timezone

from beers.models import Release

release = Release.objects.get(name=release_name)
thirty_days_ago = timezone.now() - timedelta(days=30)

beers = list(
    release.beer.filter(
        verified_match=False, untpd_id__isnull=False, created_at__gte=thirty_days_ago
    ).exclude(untpd_name="")
)

for beer in beers:
    print(f"{beer.vmp_name}")

    print(f"{beer.untpd_name}")

    user_input = input("Is this a verified match? (y/n): ").strip().lower()

    if user_input == "y":
        beer.verified_match = True
        beer.save()
        print("Verified match set to True.\n")
    elif user_input == "n":
        beer.untpd_id = None
        beer.untpd_name = None
        beer.untpd_url = None
        beer.verified_match = False
        beer.prioritize_recheck = False
        beer.brewery = None
        beer.rating = None
        beer.checkins = None
        beer.style = None
        beer.description = None
        beer.abv = None
        beer.ibu = None
        beer.label_hd_url = None
        beer.label_sm_url = None
        beer.alcohol_units = None
        beer.untpd_updated = None
        beer.save()
        print("Fields reset and saved.\n")
    else:
        print("Invalid input. Skipping this beer.\n")


## Match rest from Google


In [ ]:
import re

import requests

from beers.models import Beer

API_KEY = os.getenv("GOOGLE_API_KEY")
CSE_ID = os.getenv("GOOGLE_CUSTOM_SEACH_ID")


def query_google(beer_name):
    """
    Query Google Custom Search API for the beer name on site untappd.com and return the top 5 results.
    """
    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "key": API_KEY,
        "cx": CSE_ID,
        "q": beer_name,
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(
            f"Failed to fetch results for {beer_name}. Status code: {response.status_code}"
        )
        return []

    data = response.json()
    results = []

    valid_url_pattern = re.compile(r"^https://untappd\.com/b/[^/]+/\d+$")

    for item in data.get("items", []):
        title = item.get("title")
        link = item.get("link")

        if valid_url_pattern.match(link):
            results.append((title, link))

        if len(results) == 5:
            break

    return results


beers = Beer.objects.filter(match_manually=True, active=True)

for beer in beers:
    print(f"\nProcessing beer: {beer.vmp_name}")

    results = query_google(beer.vmp_name)

    if not results:
        print("No results found. Skipping this beer.")
        continue

    print("\nTop 5 results:")
    for i, (title, link) in enumerate(results, start=1):
        print(f"{i}. {title} - {link}")

    choice = input(
        "\nEnter the number of the correct match (or press Enter to skip): "
    ).strip()

    if choice.isdigit() and 1 <= int(choice) <= len(results):
        selected_index = int(choice) - 1
        selected_title, selected_link = results[selected_index]

        beer.untpd_url = selected_link
        beer.untpd_id = selected_link.split("/")[-1]
        beer.verified_match = True
        beer.match_manually = False
        beer.save()

        print(f"Matched '{beer.vmp_name}' to '{selected_title}' ({selected_link}).")
    else:
        print("No match selected. Skipping this beer.")


## Schedule badges


In [ ]:
from datetime import timedelta

from django.utils import timezone
from django_q.models import Schedule

Schedule.objects.create(
    name="Release: " + badge_text + " - Add badges",
    func="beers.tasks.create_badges_custom",
    kwargs="products='"
    + ",".join(str(beer) for beer in release_beers)
    + "', badge_text='"
    + badge_text
    + "', badge_type='"
    + badge_type
    + "'",
    schedule_type=Schedule.ONCE,
    next_run=timezone.now() + timedelta(minutes=10),
)

# Schedule removing badges
Schedule.objects.create(
    name="Release: " + badge_text + " - Remove badges",
    func="beers.tasks.remove_badges",
    kwargs="badge_type='" + badge_type + "'",
    schedule_type=Schedule.ONCE,
    next_run=timezone.now() + timedelta(days=badge_days),
)